In [4]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO Y CARGA DE DATOS SEGUROS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import openpyxl
from IPython.display import display  # Permite visualizar múltiples tablas seguidas

# Configuración de estilos visuales para los gráficos de Seaborn/Matplotlib
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100
print("-> Todas las librerías han sido importadas con éxito.")

# Cargar los datasets consolidados desde la carpeta local 'data'
df_empleados = pd.read_csv('data/dim_empleados.csv', encoding='utf-8-sig')
df_evaluaciones = pd.read_csv('data/fact_evaluaciones.csv', encoding='utf-8-sig')

# Confirmación de carga exitosa y dimensiones en consola
print("\n=== BASES DE DATOS IMPORTADAS CORRECTAMENTE ===")
print(f"-> 'df_empleados' (Maestro de personal): {df_empleados.shape[0]} empleados.")
print(f"-> 'df_evaluaciones' (Historial de desempeño): {df_evaluaciones.shape[0]} registros por periodos.")

# Renderizar ambas tablas de forma simultánea
print("\n[AUDITORÍA] Vista previa del Maestro de Personal (df_empleados):")
display(df_empleados.head(10))

print("\n[AUDITORÍA] Vista previa del Historial de Desempeño (df_evaluaciones):")
display(df_evaluaciones.head(10))


-> Todas las librerías han sido importadas con éxito.

=== BASES DE DATOS IMPORTADAS CORRECTAMENTE ===
-> 'df_empleados' (Maestro de personal): 1200 empleados.
-> 'df_evaluaciones' (Historial de desempeño): 3600 registros por periodos.

[AUDITORÍA] Vista previa del Maestro de Personal (df_empleados):


,ID_Empleado,Edad,Departamento,Antiguedad_Meses,Sueldo_Mensual_MXN,Horas_Extra,Distancia_Planta_KM,Meses_Ultimo_Ascenso,Nivel_Ingles,Rotacion
0,EMP-0001,56,Logística,27,21408,No,40,27,Avanzado,No
1,EMP-0002,46,RRHH,69,19906,No,12,17,Básico,No
2,EMP-0003,32,Producción,25,9318,No,13,25,Intermedio,No
3,EMP-0004,25,Producción,3,9994,No,43,3,Avanzado,Sí
4,EMP-0005,38,Producción,97,9461,Sí,17,39,Intermedio,No
5,EMP-0006,56,Producción,68,9683,No,17,5,Intermedio,No
6,EMP-0007,36,Finanzas,25,46401,No,36,21,Básico,No
7,EMP-0008,40,Producción,20,9340,No,21,20,Básico,No
8,EMP-0009,28,Producción,34,11876,No,8,0,Básico,No
9,EMP-0010,28,Producción,26,9696,Sí,5,3,Básico,No



[AUDITORÍA] Vista previa del Historial de Desempeño (df_evaluaciones):


,ID_Empleado,Periodo_Evaluacion,Score_Desempeno,Score_Potencial
0,EMP-0001,Periodo_2023,4,Medio
1,EMP-0001,Periodo_2024,5,Medio
2,EMP-0001,Periodo_2025,3,Medio
3,EMP-0002,Periodo_2023,4,Medio
4,EMP-0002,Periodo_2024,4,Bajo
5,EMP-0002,Periodo_2025,4,Medio
6,EMP-0003,Periodo_2023,2,Medio
7,EMP-0003,Periodo_2024,4,Alto
8,EMP-0003,Periodo_2025,3,Alto
9,EMP-0004,Periodo_2023,3,Medio


# 2. Auditoría de Calidad y Sanitización del Dato (Data Cleaning)

Antes de iniciar cualquier proceso de modelado o análisis exploratorio, es una regla de oro auditar la salud general de las bases de datos locales. En esta fase implementamos un control estricto para asegurar la consistencia relacional y la limpieza de las estructuras de datos.

### Acciones Ejecutadas:
* **Inspección de Estructuras (`.info()`):** Verificación manual de que Pandas interpretara correctamente los tipos de datos (variables numéricas como `int64` y categóricas como `str`).
* **Validación de Completitud:** Monitoreo del volumen de registros no nulos frente al índice general para descartar la existencia de valores perdidos (`NaN` o vacíos).
* **Control de Unicidad e Integridad Referencial:** * Se validó que el `ID_Empleado` funcione como una llave primaria única en el maestro de personal.
  * Se confirmó que no existieran evaluaciones duplicadas para un mismo empleado dentro de un mismo periodo.
  * Se comprobó la inexistencia de registros "huérfanos" (IDs en el historial de desempeño que no existan en el maestro).
* **Sanitización de Cadenas de Texto (`.str.strip()`):** Limpieza algorítmica de espacios en blanco invisibles al inicio o final de las variables de texto para blindar los futuros filtros y agrupaciones contra errores de captura.

In [9]:
# ==============================================================================
# CELDA 2: AUDITORÍA Y SANITIZACIÓN DEL MAESTRO DE PERSONAL (df_empleados)
# ==============================================================================

print("=== 1. ESTRUCTURA GENERAL Y TIPOS DE DATOS ===")
# Verificar que Pandas haya interpretado correctamente cada columna (números vs texto)
print(df_empleados.info())


print("\n" + "="*60 + "\n")



print("=== 2. PROCESO DE SANITIZACIÓN DE TEXTO ===")
# Aplicar .str.strip() para asegurar que no existan espacios invisibles que arruinen los futuros filtros
columnas_texto_emp = ['Departamento', 'Horas_Extra', 'Nivel_Ingles', 'Rotacion']
for col in columnas_texto_emp:
    df_empleados[col] = df_empleados[col].astype(str).str.strip()

print("-> Sanitización de textos completada (.str.strip() aplicado con éxito).")
print("-> Conclusión: El dataset 'df_empleados' está validado, limpio y listo.")

=== 1. ESTRUCTURA GENERAL Y TIPOS DE DATOS ===
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   ID_Empleado           1200 non-null   str  
 1   Edad                  1200 non-null   int64
 2   Departamento          1200 non-null   str  
 3   Antiguedad_Meses      1200 non-null   int64
 4   Sueldo_Mensual_MXN    1200 non-null   int64
 5   Horas_Extra           1200 non-null   str  
 6   Distancia_Planta_KM   1200 non-null   int64
 7   Meses_Ultimo_Ascenso  1200 non-null   int64
 8   Nivel_Ingles          1200 non-null   str  
 9   Rotacion              1200 non-null   str  
dtypes: int64(5), str(5)
memory usage: 93.9 KB
None


=== 2. PROCESO DE SANITIZACIÓN DE TEXTO ===
-> Sanitización de textos completada (.str.strip() aplicado con éxito).
-> Conclusión: El dataset 'df_empleados' está validado, limpio y listo.


In [10]:
# ==============================================================================
# CELDA 3: AUDITORÍA Y SANITIZACIÓN DEL HISTORIAL DE DESEMPEÑO (df_evaluaciones)
# ==============================================================================

print("=== 1. ESTRUCTURA GENERAL, TIPOS DE DATOS Y VALORACIÓN DE NULOS ===")
# Validamos dimensiones, tipos de datos e información no nula de un solo golpe
print(df_evaluaciones.info())


print("\n" + "="*60 + "\n")


print("=== 2. CONTROL DE DUPLICADOS E INTEGRIDAD REFERENCIAL ===")

# Validar que un empleado NO tenga más de una evaluación registrada en el mismo periodo
duplicados_periodo = df_evaluaciones.duplicated(subset=['ID_Empleado', 'Periodo_Evaluacion']).sum()
print(f"-> Evaluaciones duplicadas (Mismo Empleado + Mismo Periodo): {duplicados_periodo}")

# Integridad: Confirmar que no existan IDs "fantasma" en evaluaciones que no figuren en el maestro
ids_huerfanos = (~df_evaluaciones['ID_Empleado'].isin(df_empleados['ID_Empleado'])).sum()
print(f"-> Registros de evaluaciones huérfanos (ID inexistente en maestro): {ids_huerfanos}")


print("\n" + "="*60 + "\n")


print("=== 3. PROCESO DE SANITIZACIÓN DE TEXTO ===")
# Remover espacios en blanco en las columnas categóricas de esta tabla
columnas_texto_eval = ['Periodo_Evaluacion', 'Score_Potencial']
for col in columnas_texto_eval:
    df_evaluaciones[col] = df_evaluaciones[col].astype(str).str.strip()

print("-> Sanitización de textos completada (.str.strip() aplicado con éxito).")
print("-> Conclusión: El dataset 'df_evaluaciones' está validado, limpio y listo.")

=== 1. ESTRUCTURA GENERAL, TIPOS DE DATOS Y VALORACIÓN DE NULOS ===
<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ID_Empleado         3600 non-null   str  
 1   Periodo_Evaluacion  3600 non-null   str  
 2   Score_Desempeno     3600 non-null   int64
 3   Score_Potencial     3600 non-null   str  
dtypes: int64(1), str(3)
memory usage: 112.6 KB
None


=== 2. CONTROL DE DUPLICADOS E INTEGRIDAD REFERENCIAL ===
-> Evaluaciones duplicadas (Mismo Empleado + Mismo Periodo): 0
-> Registros de evaluaciones huérfanos (ID inexistente en maestro): 0


=== 3. PROCESO DE SANITIZACIÓN DE TEXTO ===
-> Sanitización de textos completada (.str.strip() aplicado con éxito).
-> Conclusión: El dataset 'df_evaluaciones' está validado, limpio y listo.


# 3. Análisis Exploratorio de Datos (EDA) - Perfilamiento y KPIs Base

Con la calidad del dato garantizada, iniciamos la fase analítica pura. El objetivo de esta sección es establecer una **línea base** del comportamiento de la plantilla de la planta a nivel global, utilizando métricas de lectura directa e indicadores clave de negocio (KPIs) de Recursos Humanos.

### Métricas Estadísticas Seleccionadas:
Para evitar ruido visual y asegurar un reporte con alto sentido práctico para la toma de decisiones gerenciales, nos enfocamos exclusivamente en cuatro pilares de tendencia central y límites reales:
1. **Mínimo y Máximo:** Definen los rangos reales y límites físicos de la población (edades, sueldos y antigüedades operativas).
2. **Media (Promedio):** Nos otorga el comportamiento central equilibrado de las variables cuantitativas.
3. **Moda:** Identifica el valor más frecuente y típico dentro de la planta (por ejemplo, el tabulador salarial más común o la edad predominante en las líneas de producción).

### Indicadores de Negocio (KPIs Globales):
* **Distribución de Headcount:** Conteo absoluto del volumen de personal asignado a cada departamento operativo.
* **Tasa Global de Rotación Voluntaria:** El indicador clave de rendimiento (KPI) que mide el porcentaje de bajas acumuladas al cierre del ciclo.
* **Incidencia de Horas Extra:** Monitoreo del porcentaje de personal que extiende su jornada laboral, sirviendo como variable crítica de desgaste o saturación operativa.

In [19]:
# ==============================================================================
# CELDA 4: ANÁLISIS EXPLORATORIO DE DATOS (EDA) - ESTADÍSTICOS Y KPIs BASE
# ==============================================================================

# 1. SELECCIÓN DE VARIABLES NUMÉRICAS CLAVE
variables_num = ['Edad', 'Antiguedad_Meses', 'Sueldo_Mensual_MXN', 'Distancia_Planta_KM', 'Meses_Ultimo_Ascenso']
df_num = df_empleados[variables_num]

print("=== 1. COMPORTAMIENTO DE LAS VARIABLES CUANTITATIVAS ===")
# Creamos la tabla personalizada con las métricas exactas solicitadas
tabla_estadistica = pd.DataFrame({
    'Mínimo': df_num.min(),
    'Máximo': df_num.max(),
    'Media (Promedio)': df_num.mean(),
    'Moda': df_num.mode().iloc[0]  # Extrae el valor más frecuente
})
display(tabla_estadistica.round(2))


print("\n" + "="*60 + "\n")


print("=== 2. DISTRIBUCIÓN DEL HEADCOUNT POR DEPARTAMENTO ===")
# Conocer el volumen absoluto de personal en cada área de la planta
print(df_empleados['Departamento'].value_counts())


print("\n" + "="*60 + "\n")


print("=== 3. INDICADORES CLAVE DE RECURSOS HUMANOS (KPIs GLOBAL) ===")

# Tasa base global de Rotación Voluntaria (expresada en porcentaje)
tasa_rotacion = df_empleados['Rotacion'].value_counts(normalize=True).round(4) * 100
print("--- Estatus de Rotación Global (%) ---")
print(tasa_rotacion)

# Porcentaje de la plantilla total que genera tiempo extra
porcentaje_horas_x = df_empleados['Horas_Extra'].value_counts(normalize=True).round(4) * 100
print("\n--- Personal con Horas Extra (%) ---")
print(porcentaje_horas_x)

=== 1. COMPORTAMIENTO DE LAS VARIABLES CUANTITATIVAS ===


,Mínimo,Máximo,Media (Promedio),Moda
Edad,18,59,38.66,50
Antiguedad_Meses,1,119,58.65,47
Sueldo_Mensual_MXN,9005,57968,20161.09,9416
Distancia_Planta_KM,1,49,25.24,45
Meses_Ultimo_Ascenso,0,60,24.96,25




=== 2. DISTRIBUCIÓN DEL HEADCOUNT POR DEPARTAMENTO ===
Departamento
Producción    693
Ingeniería    198
Logística     148
Finanzas       92
RRHH           69
Name: count, dtype: int64


=== 3. INDICADORES CLAVE DE RECURSOS HUMANOS (KPIs GLOBAL) ===
--- Estatus de Rotación Global (%) ---
Rotacion
No    82.0
Sí    18.0
Name: proportion, dtype: float64

--- Personal con Horas Extra (%) ---
Horas_Extra
No    67.17
Sí    32.83
Name: proportion, dtype: float64


# 4. Segmentación Avanzada y Análisis de Factores de Riesgo

Una vez establecida la línea base global de la planta, el análisis descriptivo promedio puede ocultar problemáticas severas concentradas en áreas específicas. En esta sección implementamos un análisis de segmentación multidimensional mediante **Tablas Dinámicas (`.pivot_table()`)** para aislar las variables críticas que impactan directamente al negocio.

### Dimensiones de Análisis Estratégico:
1. **Análisis de Vulnerabilidad por Departamento:** Cuantificamos el volumen absoluto de personal contra su respectiva tasa de rotación por área operativa. Esto permite identificar si la fuga de talento es un problema generalizado o si está focalizado en sectores específicos (como manufactura o logística).
2. **El Factor de Desgaste (Horas Extra):** Cruzamos la incidencia de tiempo extra contra el estatus de bajas de la planta. El objetivo es comprobar de forma directa si la sobrecarga laboral actúa como un detonante crítico de deserción voluntaria (*burnout*).
3. **Auditoría de Competitividad Salarial:** Evaluamos de forma comparativa la media del sueldo mensual de los empleados activos frente al sueldo que percibía el personal que decidió abandonar la organización. Esta métrica identifica si existen desviaciones o brechas económicas insostenibles dentro de los tabuladores de cada departamento.

In [20]:
# ==============================================================================
# CELDA 5: SEGMENTACIÓN AVANZADA - TABLAS DINÁMICAS DE ROTACIÓN Y NEGOCIO
# ==============================================================================

# Para facilitar el cálculo de porcentajes en las tablas dinámicas, 
# creamos una columna temporal numérica donde 'Sí' = 1 y 'No' = 0
df_empleados['Rotacion_Num'] = df_empleados['Rotacion'].map({'Sí': 1, 'No': 0})

print("=== 1. TASA DE ROTACIÓN Y VOLUMEN POR DEPARTAMENTO ===")
# Cruzamos el conteo de empleados y el promedio de rotación por área (multiplicado por 100 para porcentaje)
tabla_depto = df_empleados.pivot_table(
    values='Rotacion_Num',
    index='Departamento',
    aggfunc=['count', 'mean']
)
# Renombramos columnas para que el reporte sea ejecutivo y claro
tabla_depto.columns = ['Total_Empleados', 'Tasa_Rotacion_%']
tabla_depto['Tasa_Rotacion_%'] = (tabla_depto['Tasa_Rotacion_%'] * 100).round(2)
display(tabla_depto)


print("\n" + "="*75 + "\n")


print("=== 2. EL EFECTO INVERNADERO: HORAS EXTRA VS ROTACIÓN ===")
# Analizamos si la sobrecarga de trabajo (Horas Extra) está directamente ligada a la fuga de talento
tabla_horas = df_empleados.pivot_table(
    values='Rotacion_Num',
    index='Horas_Extra',
    aggfunc=['count', 'mean']
)
tabla_horas.columns = ['Total_Empleados', 'Tasa_Rotacion_%']
tabla_horas['Tasa_Rotacion_%'] = (tabla_horas['Tasa_Rotacion_%'] * 100).round(2)
display(tabla_horas)


print("\n" + "="*75 + "\n")


print("=== 3. COMPARATIVA SALARIAL: ¿SE VAN POR DINERO? ===")
# Evaluamos el sueldo mensual promedio de los empleados activos vs los que ya rotaron por área
tabla_sueldos = df_empleados.pivot_table(
    values='Sueldo_Mensual_MXN',
    index='Departamento',
    columns='Rotacion',
    aggfunc='mean'
).round(2)
# Renombramos las columnas del cruce para evitar confusiones
tabla_sueldos.columns = ['Sueldo_Promedio_Activos (No)', 'Sueldo_Promedio_Fugados (Sí)']
display(tabla_sueldos)

# Eliminamos la columna numérica temporal para mantener limpio nuestro DataFrame maestro
df_empleados.drop(columns=['Rotacion_Num'], inplace=True)

=== 1. TASA DE ROTACIÓN Y VOLUMEN POR DEPARTAMENTO ===


,Total_Empleados,Tasa_Rotacion_%
Departamento,,
Finanzas,92,9.78
Ingeniería,198,11.62
Logística,148,17.57
Producción,693,21.07
RRHH,69,17.39




=== 2. EL EFECTO INVERNADERO: HORAS EXTRA VS ROTACIÓN ===


,Total_Empleados,Tasa_Rotacion_%
Horas_Extra,,
No,806,10.42
Sí,394,33.50




=== 3. COMPARATIVA SALARIAL: ¿SE VAN POR DINERO? ===


,Sueldo_Promedio_Activos (No),Sueldo_Promedio_Fugados (Sí)
Departamento,,
Finanzas,39979.70,38671.67
Ingeniería,39709.17,39319.22
Logística,20330.95,19762.15
Producción,12060.72,11540.85
RRHH,20264.09,20173.42


# 5. Integración de Datos (Data Merge) y Preparación del Historial

Para profundizar en las causas de la rotación, no basta con analizar una "fotografía" estática del presente. Es imperativo conectar el desenlace del empleado con su comportamiento histórico. 

En esta sección realizamos una **unión de datos (Merge/Join)** utilizando una estrategia de *Left Join* tomando como base el historial de evaluaciones. La llave primaria de conexión es el `ID_Empleado`.

### Objetivos de la Integración:
* Vincular cada registro anual de desempeño (2023, 2024, 2025) con las variables sociodemográficas y de control del empleado (Sueldo, Departamento, Horas Extra, Rotación).
* Conservar la estructura longitudinal para analizar cómo el cambio en el desempeño o el estancamiento laboral preceden a la decisión de abandonar la planta.

In [21]:
# ==============================================================================
# CELDA 6: INTEGRACIÓN DE DATOS (MERGE) - HISTORIAL + MAESTRO DE EMPLEADOS
# ==============================================================================

# Unimos df_evaluaciones con df_empleados usando el ID_Empleado como llave común.
# Usamos 'left' para asegurar que mantenemos todo el registro histórico cronológico.
df_historico_completo = pd.merge(
    df_evaluaciones, 
    df_empleados, 
    on='ID_Empleado', 
    how='left'
)

print("=== 1. VERIFICACIÓN DE LA UNIÓN DE DATOS ===")
print(f"Dimensiones del historial original: {df_evaluaciones.shape}")
print(f"Dimensiones del nuevo DataFrame integrado: {df_historico_completo.shape}\n")

print("=== 2. VISTA PREVIA DEL HISTORIAL INTEGRADO ===")
# Ordenamos por empleado y año para observar la secuencia temporal de cada persona
display(df_historico_completo.sort_values(by=['ID_Empleado', 'Anio_Evaluacion']).head(6))

=== 1. VERIFICACIÓN DE LA UNIÓN DE DATOS ===
Dimensiones del historial original: (3600, 4)
Dimensiones del nuevo DataFrame integrado: (3600, 13)

=== 2. VISTA PREVIA DEL HISTORIAL INTEGRADO ===


KeyError: 'Anio_Evaluacion'